In [ ]:
import openap
import pandas as pd

from prc_fuel.features import flight_progress
from prc_fuel.load import load

df = load("train")
list_openap = set(openap.prop.available_aircraft())

list_df = df.aircraft_type.value_counts()
types_lower = list_df.index.str.lower()
known = types_lower.isin(list_openap)

list_df.to_frame("n_intervals").assign(is_in_openap=known)

All our aircraft types are in OpenAP except A306, B77L and MD11, 99 intervals (<0.1%). The OpenAP feature will just be NaN for them.

In [ ]:
openap.prop.aircraft("a320")

openap.prop.aircraft gives the masses in kg: oew (empty, ready to fly) 42600, mtow (max at takeoff) 78000, mlw (max at landing) 66000, mfc (max fuel) 24210 for the A320. The real mass is somewhere between oew and mtow and goes down as fuel burns. Careful, ceiling and cruise height are in meters.

Mass hypothesis: it goes linearly from mtow at takeoff to mlw at landing, following flight_progress. flight_progress is clipped to [0, 1] because some intervals are on the ground, before takeoff or after landing (~7% of flights in the exploration). This overestimates short flights, which don't take off at mtow.

In [ ]:
mtow = {t: openap.prop.aircraft(t.lower())["mtow"] for t in list_df.index[known]}
mlw = {t: openap.prop.aircraft(t.lower())["mlw"] for t in list_df.index[known]}

ac_mtow = df.aircraft_type.map(mtow)
ac_mlw = df.aircraft_type.map(mlw)
progress = flight_progress(df).clip(0, 1)

mass = ac_mtow + progress * (ac_mlw - ac_mtow)
mass.describe()

Mass goes from ~58000 to 560000 kg. Median ~77000 (A320 family, ~60% of intervals), then a jump to ~210000 at Q3 (A359, B788...). Two groups of aircraft, not an error. 99 NaN for the unknown types.

In [ ]:
from openap import aero

tas_ms = aero.mach2tas(0.78, 36000 * aero.ft)
tas_ms / aero.kts

~447 kt at mach 0.78 and 36000 ft. mach2tas wants the altitude in meters and gives m/s, hence aero.ft (meters per foot) and aero.kts (m/s per knot).

In [ ]:
from prc_fuel.clean import mask_bad_values

masked = mask_bad_values(df)
tas = aero.mach2tas(masked.avg_mach, masked.avg_altitude * aero.ft) / aero.kts
tas.describe()

TAS from mach on the masked data (mach 0 already NaN). Median ~446 kt, close to the ~440 kt groundspeed median, so wind roughly cancels out on average. Min ~29 kt is the 0.05 mach on the ground. Max 568 is high, probably a cruise mach with an average altitude pulled down by a descent. Only ~63% of intervals have a mach.

For the others, fallback speeds in order of reliability: TAS from mach (already an airspeed), then avg_groundspeed (measured on many points, but includes the wind), then the speed from the distance between the bounds (only two points, a few minutes off in time, straight line).

In [ ]:
from prc_fuel.features import duration_s, interval_distance

dist_speed = interval_distance(masked) / (duration_s(masked) / 3600) / 1.852

speed = tas.fillna(masked.avg_groundspeed).fillna(dist_speed)
speed.isna().mean()

3.8% left without speed, the intervals without positions. From ~63% to ~96% of intervals with a speed.

In [ ]:
speed.describe()

Median barely moves (447 kt), Q3 goes from 456 to 479: the added speeds are ground speeds and include the strong tailwinds seen on groundspeed. But min is 0 and max 799.5, just under the 800 kt mask, so checking where the extremes come from.

In [ ]:
low = speed < 100
high = speed > 700

source = (
    pd.Series("dist", index=speed.index)
    .mask(masked.avg_groundspeed.notna(), "gs")
    .mask(tas.notna(), "tas")
)
source[low].value_counts(), source[high].value_counts()

Below 100 kt: 3221 intervals, mostly from mach (2658, very low mach on the ground, kept) and 529 from the distance, 500 of them exactly 0 (both bounds at the same position, impossible in flight). Above 700 kt: 77, all from the distance. So only the distance speed is a problem: kept only when > 0 and <= 700 kt (real groundspeed max is 682).

In [ ]:
dist_speed = dist_speed.where((dist_speed > 0) & (dist_speed <= 700))
speed = tas.fillna(masked.avg_groundspeed).fillna(dist_speed)
speed.describe()

~580 more NaN and max under 700 kt.

In [ ]:
import numpy as np
from openap import FuelFlow

from prc_fuel.features import implied_vr

vs = implied_vr(masked).fillna(0)
flow = pd.Series(np.nan, index=masked.index)

for t in list_df.index[known]:
    rows = masked.aircraft_type == t
    ff = FuelFlow(t.lower(), use_synonym=True)
    flow[rows] = ff.enroute(
        mass=mass[rows].values,
        tas=speed[rows].values,
        alt=masked.avg_altitude[rows].values,
        vs=vs[rows].values,
    )

openap_fuel = flow * duration_s(masked)
openap_fuel.describe()

One FuelFlow per aircraft type. enroute gives kg/s, times the duration gives kg. vs is implied_vr, 0 (cruise) when missing. use_synonym=True because A21N, B763, A318 and B39M miss some OpenAP models and crash without it (it then uses a close type, e.g. a20n for a21n).

Median ~250 kg, mean ~760 kg against ~562 kg for the real fuel_kg (LightGBM's starting score), so it overestimates by about a third, as expected with a mass starting at mtow. Wide spread because of the mix of narrow and wide bodies and interval lengths.

In [ ]:
ok = openap_fuel.notna()
pd.concat([openap_fuel, masked.fuel_kg], axis=1)[ok].corr(method="spearman")

Spearman 0.93 between the OpenAP estimate and fuel_kg, the strongest link with the target so far.

In [ ]:
flow_openap = openap_fuel / duration_s(masked)
flow_real = masked.fuel_kg / duration_s(masked)

pd.Series(
    {
        t: flow_openap[masked.aircraft_type == t].corr(
            flow_real[masked.aircraft_type == t], method="spearman"
        )
        for t in ["A20N", "A320", "A359"]
    }
)

Run #14 with openap_fuel: val 298.0, clean val 169.5, rank 242.6, no better than #13. The 0.93 came from duration and aircraft size, which the model already had. Comparing fuel rates within a type, OpenAP only gets 0.35-0.6, so it says little about which interval burns more. Mass is the weakest assumption, next try: take-off mass depending on route distance.